# 03 - Share an auth session with the B2C CLI

The Python SDK and the TypeScript B2C CLI read/write the **same**
`auth-sessions.json` store. This example writes a fixture in the exact CLI format
and reads it back from Python -- proving byte-compatible interop -- then writes a
session from Python and shows the camelCase on-disk layout.

Everything happens in an isolated temp dir; the real user store is never touched.

Public API: `find_auth_session`, `list_auth_sessions`, `save_auth_session`,
`AuthSession`, `FileAuthSessionBackend`.

In [ ]:
# --- Offline, credential-free setup -------------------------------------------
# Everything below runs with NO real network and NO real credentials. HTTP is
# mocked with respx, all state lives in a throwaway temp dir, and the auth-token
# caches are reset -- mirroring the SDK's own test harness (tests/conftest.py).
import base64
import json
import os
import tempfile
import time
from pathlib import Path

import httpx
import respx

from b2c_tooling_sdk.auth.oauth import reset_oauth_cache_for_testing
from b2c_tooling_sdk.auth.oauth_implicit import reset_implicit_cache_for_testing
from b2c_tooling_sdk.auth.oauth_pkce import reset_pkce_cache_for_testing
from b2c_tooling_sdk.auth.session_store import (
    FileAuthSessionBackend,
    set_auth_session_backend,
)

_tmp = Path(tempfile.mkdtemp(prefix="b2c-nb-"))
(_tmp / "data").mkdir(parents=True, exist_ok=True)
(_tmp / "config").mkdir(parents=True, exist_ok=True)

# Point every config/data dir at the temp dir so we never touch a real user store.
os.environ["XDG_DATA_HOME"] = str(_tmp / "data")
os.environ["XDG_CONFIG_HOME"] = str(_tmp / "config")
os.environ["LOCALAPPDATA"] = str(_tmp / "data")
os.environ.pop("B2C_CONFIG_DIR", None)

# Reset the module-level OAuth token caches for deterministic runs.
reset_oauth_cache_for_testing()
reset_pkce_cache_for_testing()
reset_implicit_cache_for_testing()

# Install a temp-dir file-backed auth-session store as the default.
set_auth_session_backend(FileAuthSessionBackend(_tmp / "store"))
print("Isolated temp dir:", _tmp)

## Point the store at a temp data dir

We install a `FileAuthSessionBackend` rooted at a temp directory that stands in
for the CLI's data dir.

In [ ]:
from b2c_tooling_sdk.auth.session_store import (
    AuthSession,
    FileAuthSessionBackend,
    find_auth_session,
    list_auth_sessions,
    save_auth_session,
    set_auth_session_backend,
)

data_dir = _tmp / "cli-data"
data_dir.mkdir(parents=True, exist_ok=True)
set_auth_session_backend(FileAuthSessionBackend(data_dir))
session_file = data_dir / "auth-sessions.json"
print("Store file:", session_file)

## Read a session written by the TypeScript CLI

This is the exact camelCase JSON the CLI persists. Python reads it into a typed
`AuthSession` with snake_case attributes.

In [ ]:
ts_written = {
    "version": 1,
    "sessions": [
        {
            "clientId": "abc-123",
            "flow": "pkce",
            "accessToken": "tok",
            "refreshToken": "refresh",
            "sub": "user@example.com",
            "expiresAt": "2099-01-01T00:00:00.000Z",
            "scopes": ["sfcc.jobs"],
            "accountManagerHost": "account.demandware.com",
            "lastUsedAt": "2025-01-01T00:00:00.000Z",
        }
    ],
}
session_file.write_text(json.dumps(ts_written, indent=2), encoding="utf-8")

session = find_auth_session("abc-123")
assert session is not None
print("flow                :", session.flow)
print("access_token        :", session.access_token)
print("refresh_token       :", session.refresh_token)
print("sub                 :", session.sub)
print("scopes              :", session.scopes)
print("account_manager_host:", session.account_manager_host)

## Write a session from Python (CLI-readable)

`save_auth_session` writes camelCase keys and stamps `lastUsedAt`, so a session
created here is readable by the CLI.

In [ ]:
save_auth_session(
    AuthSession(
        client_id="python-client",
        flow="client-credentials",
        access_token="tok-from-python",
        scopes=["sfcc.catalogs"],
    )
)

on_disk = json.loads(session_file.read_text(encoding="utf-8"))
print(json.dumps(on_disk, indent=2))

client_ids = sorted(s.client_id for s in list_auth_sessions())
print("client ids in store:", client_ids)
assert "python-client" in client_ids and "abc-123" in client_ids

## Recap

- Python read a CLI-written session verbatim (camelCase -> snake_case attrs).
- Python wrote a session back in the same camelCase format the CLI expects.
- Both tools can share one `auth-sessions.json` -- log in with one, use the other.